# Notebook 13: Tree-Based Classifiers — Baseline (Frozen Embeddings)

## Purpose
Evaluate four tree-based ensemble classifiers on frozen `all-MiniLM-L6-v2`
embeddings as an alternative to the MLP head explored in Notebooks 09-13.
Tree-based classifiers require no gradient-based training, no batch scheduling,
and no early stopping — they fit directly on the full embedding matrix.

## Classifiers
| Classifier | Imbalance Strategy |
|---|---|
| Random Forest (RF) | `class_weight='balanced'` |
| Extra Trees (ET) | `class_weight='balanced'` |
| XGBoost (XGB) | `scale_pos_weight=102` |
| LightGBM (LGBM) | `is_unbalance=True` |

## Baseline
Real training data only — no augmentation. Direct comparison to:
- LR baseline: F1=0.152
- MLP-1 baseline: F1=0.270
- MLP-2 baseline: F1=0.207

## Evaluation
- Threshold optimized on calibration set — identical to Notebooks 09-12
- Metrics: Precision, Recall, F1 vs MLP baselines
- DET curves: all four tree classifiers overlaid

In [ ]:
# -----------------------------------------------
# Imports and setup
# -----------------------------------------------
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    det_curve,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

# -----------------------------------------------
# Load frozen embeddings and splits
# -----------------------------------------------
X_train = np.load("data/processed/embeddings/X_train.npy")
y_train = np.load("data/processed/embeddings/y_train.npy")
X_cal   = np.load("data/processed/embeddings/X_cal.npy")
y_cal   = np.load("data/processed/embeddings/y_cal.npy")
X_test  = np.load("data/processed/embeddings/X_test.npy")
y_test  = np.load("data/processed/embeddings/y_test.npy")

print(f"Train: {X_train.shape} | Positives: {y_train.sum()}")
print(f"Cal:   {X_cal.shape}   | Positives: {y_cal.sum()}")
print(f"Test:  {X_test.shape}  | Positives: {y_test.sum()}")

# Imbalance ratio for XGBoost scale_pos_weight
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
SCALE_POS_WEIGHT = n_neg / n_pos
print(f"\nImbalance ratio: {SCALE_POS_WEIGHT:.1f}:1")

# -----------------------------------------------
# Load MLP baseline metrics for comparison
# -----------------------------------------------
with open("data/results/metrics_09_baseline.json") as f:
    mlp_baseline = json.load(f)

print("\nMLP baselines (Notebook 09):")
for name, m in mlp_baseline.items():
    print(f"  {name}: F1={m['f1']:.3f} | "
          f"P={m['precision']:.3f} | R={m['recall']:.3f}")